In [3]:

import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def recall_at_k(retrieved_ids, relevant_ids, k=5):
    relevant = set(relevant_ids)

    if not relevant:
        return 0.0

    retrieved = set(retrieved_ids[:k])
    return len(retrieved & relevant) / len(relevant)


def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant = set(relevant_ids)

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant:
            return 1.0 / rank

    return 0.0


def evaluate_retrieval(retriever, questions, method="hybrid"):
    """
    questions: list of dictionaries, each with:
      question: str
      relevant_chunk_ids: list[str]
    """
    rows = []

    for item in questions:
        query = item["question"]
        relevant_ids = item["relevant_chunk_ids"]

        if method == "bm25":
            results = retriever.search_bm25(query, top_k=10)
        elif method == "dense":
            results = retriever.search_dense(query, top_k=10)
        elif method == "hybrid":
            results = retriever.search_hybrid(
                query, top_k=10, candidate_k=20
            )
        elif method == "hybrid_rerank":
            candidates = retriever.search_hybrid(
                query, top_k=20, candidate_k=30
            )
            results = retriever.rerank(query, candidates, top_k=10)
        else:
            raise ValueError(f"Unknown method: {method}")

        retrieved_ids = [
            result["chunk_id"] for result in results
        ]

        rows.append({
            "question": query,
            "method": method,
            "recall_at_5": recall_at_k(
                retrieved_ids, relevant_ids, 5
            ),
            "recall_at_10": recall_at_k(
                retrieved_ids, relevant_ids, 10
            ),
            "mrr": reciprocal_rank(
                retrieved_ids, relevant_ids
            )
        })

    return pd.DataFrame(rows)


def save_metrics(dataframe, path="results/metrics.csv"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(path, index=False)


def evaluate_verifier(true_labels, predicted_labels):
    labels = ["supported", "contradicted", "uncertain"]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        labels=labels,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


def load_questions(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)
